In [ ]:
data_dir = "DATA_DIRECTORY"

In [1]:
import os
os.environ["KERAS_BACKEND"] = "jax"

import keras
import jax
import jax.numpy as jnp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib

SEED = 0

In [ ]:
x_train = pd.read_hdf(os.path.join(data_dir, "Datasets", "x_train_scaled.h5"))
x_val = pd.read_hdf(os.path.join(data_dir, "Datasets", "x_val_scaled.h5"))

scaler = joblib.load(os.path.join(data_dir, "Datasets", "scaler.save"))

y_train = pd.read_hdf(os.path.join(data_dir, "Datasets", "y_train.h5"))
y_val = pd.read_hdf(os.path.join(data_dir, "Datasets", "y_val.h5"))

w_train_full = np.load(os.path.join(data_dir, "Datasets", "w_train.npy"))
w_val_full = np.load(os.path.join(data_dir, "Datasets", "w_val.npy"))

w_train_base = np.ones_like(w_train_full)
w_val_base = np.ones_like(w_val_full)

In [4]:
n_1 = 256; n_2 = 128; n_3 = 256
n_4 = 256; n_5 = 128; n_6 = 256

noise_std = 0.05
rate = 0.1
batch_size = 2**12

In [5]:
base_features = np.arange(6)

In [6]:
quantile = 0.5 # 0.05, 0.25, 0.5, 0.75, 0.95

In [ ]:
def relu_quantile(q):
    q = float(q)
    def loss(y_true, y_pred):
        mask = jnp.isfinite(y_true)
        y_hat = keras.ops.relu(y_pred)
        err = y_true - y_hat
        pb = jnp.where(err >= 0, q * err, (q - 1.0) * err)
        pb = jnp.where(mask, pb, 0.0)
        n = jnp.sum(mask)
        return jnp.sum(pb) / jnp.maximum(n, 1)
    return loss

def relu_pearson(y_true, y_pred):
    mask = jnp.isfinite(y_true)
    n = jnp.sum(mask)
    n = jnp.maximum(n, 1)
    y_hat = keras.ops.relu(y_pred)
    y_t = jnp.where(mask, y_true, 0.0)
    y_p = jnp.where(mask, y_hat, 0.0)
    mean_t = jnp.sum(y_t) / n
    mean_p = jnp.sum(y_p) / n
    dt = jnp.where(mask, y_t - mean_t, 0.0)
    dp = jnp.where(mask, y_p - mean_p, 0.0)
    num = jnp.sum(dt * dp)
    den = jnp.sqrt(jnp.sum(dt * dt) * jnp.sum(dp * dp))
    return num / jnp.maximum(den, 1e-12)

In [ ]:
n_base = 6
n_full = x_train.shape[1]

seed_d = 1 + SEED

# Base branch
inp_base = keras.layers.Input(shape=(n_base,), name='base_inputs')

base_n1 = keras.layers.GaussianNoise(noise_std, seed=111+seed_d, name='BaseNoise1')(inp_base)
base_d1 = keras.layers.Dense(n_1, activation='gelu', name='BaseDense1')(base_n1)
base_o1 = keras.layers.Dropout(rate, seed=101+seed_d, name='BaseDropout1')(base_d1)

base_n2 = keras.layers.GaussianNoise(noise_std, seed=112+seed_d, name='BaseNoise2')(base_o1)
base_d2 = keras.layers.Dense(n_2, activation='gelu', name='BaseDense2')(base_n2)
base_o2 = keras.layers.Dropout(rate, seed=102+seed_d, name='BaseDropout2')(base_d2)

base_n3 = keras.layers.GaussianNoise(noise_std, seed=113+seed_d, name='BaseNoise3')(base_o2)
base_d3 = keras.layers.Dense(n_3, activation='gelu', name='BaseDense3')(base_n3)
base_o3 = keras.layers.Dropout(rate, seed=103+seed_d, name='BaseDropout3')(base_d3)

base_out = keras.layers.Dense(9, name='base_out')(base_o3)

# Disturbed branch
inp_full = keras.layers.Input(shape=(n_full,), name='full_inputs')
base_out_sg = keras.ops.stop_gradient(base_out)
inp_dist = keras.layers.Concatenate()([inp_full, base_out_sg])

l1_reg = 1e-4
dist_n1 = keras.layers.GaussianNoise(noise_std, seed=211+seed_d, name='DisturbedNoise1')(inp_dist)
dist_d1 = keras.layers.Dense(
    n_4,
    activation='gelu',
    kernel_regularizer=keras.regularizers.l1(l1_reg),
    name='DisturbedDense1'
)(dist_n1)
dist_o1 = keras.layers.Dropout(rate, seed=201+seed_d, name='DisturbedDropout1')(dist_d1)

dist_n2 = keras.layers.GaussianNoise(noise_std, seed=212+seed_d, name='DisturbedNoise2')(dist_o1)
dist_d2 = keras.layers.Dense(n_5, activation='gelu', name='DisturbedDense2')(dist_n2)
dist_o2 = keras.layers.Dropout(rate, seed=202+seed_d, name='DisturbedDropout2')(dist_d2)

dist_n3 = keras.layers.GaussianNoise(noise_std, seed=213+seed_d, name='DisturbedNoise3')(dist_o2)
dist_d3 = keras.layers.Dense(n_6, activation='gelu', name='DisturbedDense3')(dist_n3)
dist_o3 = keras.layers.Dropout(rate, seed=203+seed_d, name='DisturbedDropout3')(dist_d3)

dist_out = keras.layers.Dense(9, name='dist_out')(dist_o3)

def add_stopgrad_base(inputs):
    base, dist = inputs
    return keras.ops.stop_gradient(base) + dist

final_out = keras.layers.Lambda(add_stopgrad_base, name='final_out')([base_out, dist_out])

model = keras.Model(
    inputs=[inp_base, inp_full],
    outputs=[base_out, final_out]
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss={'base_out': relu_quantile(quantile), 'final_out': relu_quantile(quantile)},
    loss_weights={'base_out': 1, 'final_out': 1},
    metrics={'base_out': [relu_pearson], 'final_out': [relu_pearson]},
    steps_per_execution=128
)

model.summary()

early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    [x_train.iloc[:, base_features], x_train],
    [y_train, y_train],
    sample_weight=[w_train_base, w_train_full],
    batch_size=batch_size,
    validation_data=(
        [x_val.iloc[:, base_features], x_val],
        [y_val, y_val],
        [w_val_base, w_val_full]
    ),
    epochs=50,
    callbacks=[early_stopping]
)

In [ ]:
model.save(os.path.join(data_dir, f"model_{int(quantile*100)}.keras"))